In [1]:
#pd.set_option('display.max_rows', None)
import numpy as np
import pandas as pd
import bokeh
import hvplot.pandas
import holoviews as hv
import bokeh.palettes
from bokeh.plotting import figure, show, output_notebook

import neuprint

import importlib
import lib as cl
from lib import syn_specs
from neuprint import SynapseCriteria

import matplotlib.pyplot as plt
from matplotlib.patches import Patch



import napari
from aicsimageio import AICSImage
from skimage import measure
import tifffile as tf
import xarray as xr
from tifffile import tifffile
import tifftools
import os
from os.path import sep
from skimage import io
from PIL import Image
import imageio
from skimage.measure import regionprops_table
import seaborn as sns
import czifile
import math
from pathlib import Path
import re
from bokeh.io import output_notebook, show as bokeh_show
output_notebook(hide_banner=True)
show = bokeh_show 

import itertools
from scipy.optimize import curve_fit, least_squares



/Users/fisherguest/miniconda3/envs/sansachen_czi/lib/python3.11/site-packages/pydantic/_migration.py:283: UserWarning: `pydantic.error_wrappers:ValidationError` has been moved to `pydantic:ValidationError`.
  warnings.warn(f'`{import_path}` has been moved to `{new_location}`.')


<font size="7">all delta7-->each delta7</font>

In [ ]:
data_dir = Path("csv_25summer") / "csv_all_d7_upstreams"

csv_files = sorted(data_dir.glob('*.csv'))

all_d7_upstream_results = {
    f.stem: pd.read_csv(f)
    for f in csv_files
}
'''
for name, df in all_d7_upstream_results.items():
    print(name, df.shape)
'''
len(all_d7_upstream_results)

42

,Glomerulus,SynapseCount
0,PB(R1),2
1,PB(R2),49
2,PB(R3),27
3,PB(R4),31
4,PB(R5),24
5,PB(R6),31
6,PB(R7),32
7,PB(R8),23
8,PB(R9),0
9,PB(L1),45


In [7]:
def reorder_pb_glomeruli_for(tag: str, env=None, *, verbose: bool=False, return_dict: bool=False):
    """
    Reorder rows in all_<tag>_upstream_results so they go:
      PB(L9) ... PB(L1) PB(R1) ... PB(R9)

    Creates all_<tag>_upstream_results_reordered and ALSO overwrites all_<tag>_upstream_results.
    By default prints nothing and returns nothing to avoid giant output in notebooks.

    Parameters
    ----------
    tag : {'d7','lpsp','p1-9','p19'}
    env : dict-like namespace where variables live (defaults to globals()).
    verbose : bool, if True prints a short summary and any warnings.
    return_dict : bool, if True returns the reordered dict; otherwise returns None.
    """
    # normalize tag → variable suffix
    key = tag.strip().lower().replace("p1-9", "p19")
    suffix_map = {"d7": "d7", "lpsp": "lpsp", "p19": "p19"}
    if key not in suffix_map:
        raise ValueError("tag must be one of {'d7','lpsp','p1-9'/'p19'}")
    suffix = suffix_map[key]

    in_name  = f"all_{suffix}_upstream_results"
    out_name = f"all_{suffix}_upstream_results_reordered"

    # where to look for the dict variables
    env = globals() if env is None else env
    if in_name not in env or not isinstance(env[in_name], dict):
        raise NameError(f"Expected a dict named '{in_name}' in scope.")

    dfs_dict = env[in_name]

    # desired top→bottom order
    desired = [f"PB(L{i})" for i in range(9, 0, -1)] + [f"PB(R{i})" for i in range(1, 10)]

    out, skipped = {}, []
    for name, df in dfs_dict.items():
        # find glomerulus column
        gcol = 'Glomerulus' if 'Glomerulus' in df.columns else ('glomerulus' if 'glomerulus' in df.columns else None)
        if gcol is None:
            skipped.append(name)
            continue

        tmp = df.copy()
        tmp[gcol] = tmp[gcol].astype(str)
        tmp['__ord'] = pd.Categorical(tmp[gcol], categories=desired, ordered=True)
        tmp = tmp.sort_values('__ord').drop(columns='__ord').reset_index(drop=True)
        out[name] = tmp

    # expose results under requested variable names
    env[out_name] = out
    env[in_name] = out  # overwrite original, per your request

    if verbose:
        if skipped:
            print(f"[warn] skipped {len(skipped)} items without a Glomerulus column (e.g., {skipped[:3]})")
        print(f"[ok] Reordered {len(out)} DataFrames -> '{out_name}', and overwrote '{in_name}'.")

    if return_dict:
        return out

In [8]:
reorder_pb_glomeruli_for("d7")

In [9]:
all_d7_upstream_results['delta7_1158747783_L7R2_L_upstream']

,Glomerulus,SynapseCount
0,PB(L9),13
1,PB(L8),9
2,PB(L7),27
3,PB(L6),15
4,PB(L5),43
5,PB(L4),81
6,PB(L3),83
7,PB(L2),29
8,PB(L1),45
9,PB(R1),2


In [10]:
def split_and_halve_upstream_results(tag: str, env=None, *, verbose=False, return_dicts=False):
    """
    Split all_<tag>_upstream_results into 3glo vs 2glo (by name substring),
    then produce L/R halves for each by slicing rows [9:] (R) and [:9] (L).

    Creates/overwrites in the given namespace (globals() by default):
        all_<tag>_upstream_results_3glo
        all_<tag>_upstream_results_2glo
        all_<tag>_upstream_results_3glo_halves
        all_<tag>_upstream_results_2glo_halves

    Parameters
    ----------
    tag : {'d7','lpsp','p19','p1-9'}
    env : dict-like or None, where variables live (defaults to globals()).
    verbose : bool, print a short summary if True (default False).
    return_dicts : bool, return the produced dicts if True (default False).

    Returns
    -------
    None (default) or dict with the four produced dicts if return_dicts=True.
    """
    key = tag.strip().lower().replace("p1-9", "p19")
    if key not in {"d7", "lpsp", "p19"}:
        raise ValueError("tag must be one of {'d7','lpsp','p19'/'p1-9'}")

    env = globals() if env is None else env
    in_name = f"all_{key}_upstream_results"
    if in_name not in env or not isinstance(env[in_name], dict):
        raise NameError(f"Expected a dict named '{in_name}' in scope.")

    src = env[in_name]

    # 1) split into 3glo vs 2glo by name pattern
    dict_3glo, dict_2glo = {}, {}
    for name, df in src.items():
        if ('L1L9R8_R' in name) or ('L8R1R9_L' in name):
            dict_3glo[name] = df
        else:
            dict_2glo[name] = df

    # 2) halves (rows [9:] -> R half, [:9] -> L half)
    def _halve(d):
        out = {}
        for name, df in d.items():
            n = len(df)
            mid = 9 if n >= 18 else max(n // 2, 0)
            out[f"{name}_R"] = df.iloc[mid:].reset_index(drop=True)
            out[f"{name}_L"] = df.iloc[:mid].reset_index(drop=True)
        return out

    halves_2glo = _halve(dict_2glo)
    halves_3glo = _halve(dict_3glo)

    # 3) expose in namespace
    env[f"all_{key}_upstream_results_3glo"] = dict_3glo
    env[f"all_{key}_upstream_results_2glo"] = dict_2glo
    env[f"all_{key}_upstream_results_3glo_halves"] = halves_3glo
    env[f"all_{key}_upstream_results_2glo_halves"] = halves_2glo

    if verbose:
        print(f"[ok] all_{key}_upstream_results: {len(src)} → "
              f"{len(dict_2glo)} (2glo) + {len(dict_3glo)} (3glo); "
              f"halves: {len(halves_2glo)} (2glo) + {len(halves_3glo)} (3glo).")

    if return_dicts:
        return {
            "2glo": dict_2glo,
            "3glo": dict_3glo,
            "2glo_halves": halves_2glo,
            "3glo_halves": halves_3glo,
        }
    # else return nothing (prevents Jupyter from displaying huge structures)

In [11]:
split_and_halve_upstream_results("d7")

In [12]:
all_d7_upstream_results_3glo['delta7_5813061383_L1L9R8_R_upstream']

,Glomerulus,SynapseCount
0,PB(L9),32
1,PB(L8),9
2,PB(L7),32
3,PB(L6),43
4,PB(L5),40
5,PB(L4),84
6,PB(L3),33
7,PB(L2),7
8,PB(L1),58
9,PB(R1),17


In [13]:
all_d7_upstream_results_2glo['delta7_1158747783_L7R2_L_upstream']

,Glomerulus,SynapseCount
0,PB(L9),13
1,PB(L8),9
2,PB(L7),27
3,PB(L6),15
4,PB(L5),43
5,PB(L4),81
6,PB(L3),83
7,PB(L2),29
8,PB(L1),45
9,PB(R1),2
